# 01. Data and definitions

**Project aim:** forecast Great Britain's half-hourly electricity demand one day ahead, with
uncertainty ranges, and compare the result with the forecast published by the National
Energy System Operator (NESO).

**Why this matters:** electricity is hard to store at scale, so supply has to match demand
second by second. Day-ahead demand forecasts feed into how generators and suppliers plan
their positions, and how NESO balances the system. A forecast error costs money, because
someone has to buy or sell the difference closer to real time.

**What this notebook does:**

1. Downloads NESO's historic demand data.
2. Pins down what the columns mean, because "demand" has several definitions.
3. Checks the data before anything is built on it.

Each step says what is being done and why, followed by what the data showed.

**Background reading**
- [Electricity sector in the United Kingdom](https://en.wikipedia.org/wiki/Electricity_sector_in_the_United_Kingdom): an overview of the system.
- [National Grid (Great Britain)](https://en.wikipedia.org/wiki/National_Grid_(Great_Britain)): how the transmission network works.
- [Forecasting: Principles and Practice](https://otexts.com/fpp3/): the textbook I am using for forecasting method.


## Step 1. Set up

**What:** import libraries and fix where files go.

**Why:** paths are relative to this notebook, so it runs on any machine. Raw files are
kept apart from anything derived from them, so a mistake in cleaning can never damage the
original download.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 90


## Step 2. Download the data

**What:** use NESO's data portal API to find the yearly CSV files in the
[Historic Demand Data](https://www.neso.energy/data-portal/historic-demand-data) dataset,
then save each one.

**Why the API and not a manual download:** anyone can rerun this and get the same files,
and the notebook picks up new yearly files without edits.

**Why start at 2018:** NESO's published day-ahead forecast, which is the benchmark here,
only goes back to April 2021. Starting in 2018 gives a few years of history before that,
including the COVID period, without modelling a much older system with far less solar.

**Why skip files already on disk:** repeat runs should not hit NESO's servers again. Delete
a file to fetch it fresh, which is worth doing now and then because NESO revises past
values (see below).


In [2]:
API = "https://api.neso.energy/api/3/action/package_show"
FIRST_YEAR = 2018

package = requests.get(API, params={"id": "historic-demand-data"}, timeout=60).json()["result"]

urls = {}
for resource in package["resources"]:
    name = resource["name"]
    if name.startswith("Historic Demand Data") and name[-4:].isdigit():
        year = int(name[-4:])
        if year >= FIRST_YEAR:
            urls[year] = resource["url"]

for year, url in sorted(urls.items()):
    path = RAW / f"demanddata_{year}.csv"
    if not path.exists():
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        path.write_bytes(response.content)
    print(year, f"{path.stat().st_size / 1e6:.1f} MB")

print("Licence:", package["license_title"])


2018 1.4 MB
2019 1.6 MB
2020 1.6 MB
2021 1.6 MB
2022 1.7 MB
2023 1.7 MB
2024 1.8 MB
2025 1.9 MB
2026 1.5 MB
Licence: NESO Open Data Licence


**What came back:** one CSV per year, each between about 1.4 and 1.8 MB, released under the
[NESO Open Data Licence](https://www.neso.energy/data-portal/ngeso-open-licence).

Two points from NESO's dataset description affect the rest of the project:

- The data is published **21 days in arrears**. It cannot supply yesterday's demand to a
  forecast made today, so Day 4 has to settle what a forecaster could really have known.
- Past values can be **revised**, especially solar. So these files are not exactly what
  anyone saw at the time.
